# Staged RL Phase C/D Continuation for Qwen2.5-VL

This notebook copies the uploaded code dataset into `/kaggle/working`, resolves the cached base model from the attached Kaggle dataset, selects the best Phase B checkpoint by composite score from the attached prior kernel output, and continues with Phase C then Phase D on the larger MathVista `test` split.


In [ ]:
import glob, json, os, shutil, subprocess, tarfile
from datetime import datetime

WORKDIR = "/kaggle/working/RL_GSPO_Qwen2.5VLM"
VENV_DIR = "/tmp/rl-gspo-venv"
VENV_PYTHON = f"{VENV_DIR}/bin/python"
RUN_ID = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


def write_progress(message):
    timestamp = datetime.utcnow().isoformat()
    print(f"{timestamp} {message}", flush=True)


def find_code_dataset_root():
    matches = []
    for root, _, files in os.walk("/kaggle/input"):
        file_set = set(files)
        has_packaged_tree = {"rl_gspo_qwen2_5vlm_test3.py", "staged_rl.tar", "tests.tar"}.issubset(file_set)
        has_nested_extracted_tree = (
            os.path.isfile(os.path.join(root, "staged_rl", "staged_rl", "trainer_runtime.py"))
            and os.path.isfile(os.path.join(root, "tests", "tests", "test_config.py"))
        )
        has_flat_extracted_tree = (
            os.path.isfile(os.path.join(root, "staged_rl", "trainer_runtime.py"))
            and os.path.isfile(os.path.join(root, "tests", "test_config.py"))
        )
        has_extracted_tree = has_nested_extracted_tree or has_flat_extracted_tree
        if "rl_gspo_qwen2_5vlm_test3.py" in file_set and (has_packaged_tree or has_extracted_tree):
            matches.append(root)
    if not matches:
        raise FileNotFoundError(f"Could not find attached code dataset. Visible inputs: {glob.glob('/kaggle/input/*')}")
    if len(matches) > 1:
        raise RuntimeError(f"Ambiguous code dataset attachment. Matches={matches}. Visible inputs: {glob.glob('/kaggle/input/*')}")
    return matches[0]


def find_best_phase_b_checkpoint():
    candidates = []
    for root, _, files in os.walk("/kaggle/input"):
        if "checkpoint_info.json" not in files:
            continue
        checkpoint_info_path = os.path.join(root, "checkpoint_info.json")
        normalized_path = checkpoint_info_path.replace("\\", "/")
        if "/phase_b/" not in normalized_path:
            continue
        with open(checkpoint_info_path, "r", encoding="utf-8") as handle:
            info = json.load(handle)
        metrics = info.get("metrics", {})
        composite = float(metrics.get("composite_score", float("-inf")))
        global_step = int(info.get("global_step", -1))
        candidates.append((composite, global_step, checkpoint_info_path, info))
    if not candidates:
        raise FileNotFoundError(
            f"Could not find any Phase B checkpoint_info.json under /kaggle/input. Visible inputs: {glob.glob('/kaggle/input/*')}"
        )
    composite, global_step, checkpoint_info_path, info = max(candidates, key=lambda item: (item[0], item[1]))
    checkpoint_dir = os.path.dirname(checkpoint_info_path)
    return checkpoint_dir, checkpoint_info_path, info, composite, global_step


write_progress(f"Notebook start (run_id={RUN_ID})")
CODE_DATASET_ROOT = find_code_dataset_root()
WARM_START_PATH, CHECKPOINT_INFO_PATH, SELECTED_CHECKPOINT_INFO, BEST_COMPOSITE, BEST_GLOBAL_STEP = find_best_phase_b_checkpoint()
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
shutil.copytree(CODE_DATASET_ROOT, WORKDIR)
for archive_name in ("staged_rl.tar", "tests.tar"):
    archive_path = os.path.join(WORKDIR, archive_name)
    if os.path.exists(archive_path):
        with tarfile.open(archive_path, "r") as tf:
            tf.extractall(WORKDIR)
        os.remove(archive_path)
        print("Extracted", archive_name)
for folder_name in ("staged_rl", "tests"):
    nested_path = os.path.join(WORKDIR, folder_name, folder_name)
    target_path = os.path.join(WORKDIR, folder_name)
    if os.path.isdir(nested_path):
        temp_path = f"{target_path}_flat"
        if os.path.exists(temp_path):
            shutil.rmtree(temp_path)
        shutil.move(nested_path, temp_path)
        shutil.rmtree(target_path)
        shutil.move(temp_path, target_path)
        print("Flattened", folder_name)
write_progress("Copied code dataset")
print("Using code dataset", CODE_DATASET_ROOT)
print("Selected Phase B checkpoint info", CHECKPOINT_INFO_PATH)
print("Selected Phase B checkpoint path", WARM_START_PATH)
print("Selected Phase B checkpoint global_step", BEST_GLOBAL_STEP)
print("Selected Phase B composite", BEST_COMPOSITE)
print(json.dumps(SELECTED_CHECKPOINT_INFO.get("metrics", {}), indent=2))
print("Copied code to", WORKDIR)
print(sorted(os.listdir(WORKDIR)))


In [ ]:
write_progress("Checking GPU availability")
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([
    "python3",
    "-c",
    "import torch; print('system_torch_version:', torch.__version__); print('cuda_available:', torch.cuda.is_available()); print('cuda_device_count:', torch.cuda.device_count())",
], check=True)
write_progress("GPU preflight passed")


In [ ]:
write_progress("Installing uv and creating venv")
subprocess.run(["python3", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["python3", "-m", "uv", "venv", "--seed", "--clear", VENV_DIR], check=True)
install_commands = [
    [VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--upgrade", "pip", "setuptools", "wheel"],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "numpy==1.26.4",
        "scipy==1.15.3",
        "scikit-learn==1.6.1",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "pillow",
        "packaging",
        "safetensors",
        "torchvision",
        "bitsandbytes",
        "xformers",
        "huggingface_hub>=0.34.0",
        "datasets==4.3.0",
        "transformers==4.56.2",
        "trl==0.22.2",
        "unsloth",
        "modelscope",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "vllm==0.10.2",
        "--extra-index-url",
        "https://wheels.vllm.ai/0.10.2/",
    ],
]
for command in install_commands:
    print("Running:", " ".join(command))
    subprocess.run(command, check=True, cwd=WORKDIR)
write_progress("Finished venv package installs")
print("Venv ready:", VENV_PYTHON)


In [ ]:
compat_script = """
import numpy
import scipy
import sklearn
import vllm
from vllm.sampling_params import GuidedDecodingParams
from trl import GRPOTrainer
print('numpy version:', numpy.__version__)
print('scipy version:', scipy.__version__)
print('sklearn version:', sklearn.__version__)
print('vllm version:', vllm.__version__)
print('compat imports OK')
"""
subprocess.run([VENV_PYTHON, "-c", compat_script], check=True, cwd=WORKDIR)
write_progress("Verified dependencies in venv")


In [ ]:
HARDWARE_PROFILE = "kaggle_t4"
TRAIN_SPLIT = "test"
EVAL_SPLIT = "testmini"
OUTPUT_ROOT = "outputs_staged_phase_cd_continue"
MAX_EVAL_EXAMPLES_PER_SUBSET = 4


def resolve_model_path():
    candidates = glob.glob("/kaggle/input/**/config.json", recursive=True)
    matches = []
    for config_path in candidates:
        root = os.path.dirname(config_path)
        if os.path.exists(os.path.join(root, "model.safetensors")):
            matches.append(root)
    if not matches:
        raise FileNotFoundError("Could not find model config.json + model.safetensors under /kaggle/input.")
    if len(matches) > 1:
        write_progress(f"Multiple model candidates found: {matches}")
    return matches[0]


MODEL_PATH = resolve_model_path()
PHASE_RUNS = [
    {"phase": "phase_c", "resume": None, "warm_start": True},
    {"phase": "phase_d", "resume": "best_composite", "warm_start": False},
]

if not os.path.exists(os.path.join(WARM_START_PATH, "adapter_model.safetensors")):
    raise FileNotFoundError(f"Warm-start adapter not found at {WARM_START_PATH}")
if not os.path.exists(os.path.join(MODEL_PATH, "config.json")):
    raise FileNotFoundError(f"Expected config.json under {MODEL_PATH}.")

write_progress(f"Resolved model path: {MODEL_PATH}")
write_progress(f"Warm-start checkpoint: {WARM_START_PATH}")

env = dict(os.environ)
env["PYTHONUNBUFFERED"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["UNSLOTH_USE_MODELSCOPE"] = "0"
env["TRANSFORMERS_OFFLINE"] = "0"
env["HF_HUB_OFFLINE"] = "0"
env["HF_DATASETS_OFFLINE"] = "0"

for run_spec in PHASE_RUNS:
    phase = run_spec["phase"]
    resume = run_spec["resume"]
    cmd = [
        VENV_PYTHON,
        "rl_gspo_qwen2_5vlm_test3.py",
        "--phase", phase,
        "--hardware-profile", HARDWARE_PROFILE,
        "--output-root", OUTPUT_ROOT,
        "--train-split", TRAIN_SPLIT,
        "--eval-split", EVAL_SPLIT,
        "--base-model-path", MODEL_PATH,
        "--max-eval-examples-per-subset", str(MAX_EVAL_EXAMPLES_PER_SUBSET),
    ]
    if resume:
        cmd.extend(["--resume", resume])
    if run_spec["warm_start"]:
        cmd.extend(["--warm-start-checkpoint", WARM_START_PATH])

    phase_output_dir = os.path.join(WORKDIR, OUTPUT_ROOT, phase)
    os.makedirs(phase_output_dir, exist_ok=True)
    train_log_path = os.path.join(phase_output_dir, "train_subprocess.log")
    write_progress(f"Starting phase {phase} (resume={resume or 'none'})")
    print("Running:", " ".join(cmd))
    print("Subprocess log:", train_log_path)
    with open(train_log_path, "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            cwd=WORKDIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            log_file.write(line)
            log_file.flush()
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code != 0:
        write_progress(f"Phase {phase} failed with return code {return_code}")
        print(f"Phase {phase} failed with return code {return_code}. Last 200 log lines:")
        with open(train_log_path, "r", encoding="utf-8") as log_file:
            tail_lines = log_file.readlines()[-200:]
        print("".join(tail_lines))
        raise subprocess.CalledProcessError(return_code, cmd)
    write_progress(f"Phase {phase} completed successfully")
    print(f"Phase {phase} completed successfully.")

print("All continuation phases completed successfully.")


In [ ]:
outputs_root = os.path.join(WORKDIR, OUTPUT_ROOT)
collected = []
for root, _, files in os.walk(outputs_root):
    for file_name in files:
        collected.append(os.path.join(root, file_name))
for path in sorted(collected)[-80:]:
    print(path)
